# スペクトル近似ノートブック

複数の基底スペクトル（例：3つ）を線形結合して、ターゲットスペクトルを最もよく近似する重みを求めます。

## 対応手法
1. **最小二乗法（制約なし）** — `numpy.linalg.lstsq`
2. **非負最小二乗法（NNLS）** — `scipy.optimize.nnls`（重みが0以上）
3. **混合比制約あり** — 重みの合計=1、各重みが0以上（材料の混合比など）

## 0. パッケージインストール（Google Colab 用）

In [ ]:
# Google Colab では以下が最初から使えますが、念のため確認します
import importlib, sys
for pkg in ['numpy', 'scipy', 'matplotlib']:
    if importlib.util.find_spec(pkg):
        print(f'✅ {pkg} is available')
    else:
        print(f'⬇️  Installing {pkg}...')
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

## 1. ライブラリのインポート

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import nnls, minimize

# 日本語フォント設定（Colab 用）
try:
    import subprocess
    subprocess.run(['apt-get', 'install', '-y', 'fonts-noto-cjk'], capture_output=True)
    plt.rcParams['font.family'] = 'Noto Sans CJK JP'
except Exception:
    pass

plt.rcParams['figure.dpi'] = 120
print('インポート完了')

## 2. スペクトルデータの準備

### 2-A. サンプルデータの自動生成（データがない場合）
実際のデータがある場合は **2-B** を使ってください。

In [ ]:
np.random.seed(42)

# 波長軸（例：400〜800 nm、200点）
wavelength = np.linspace(400, 800, 200)

def gaussian(x, mu, sigma, amp=1.0):
    """ガウシアン関数でスペクトルピークを模擬"""
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

# 基底スペクトル（3つ）
s1 = gaussian(wavelength, mu=480, sigma=30) + gaussian(wavelength, mu=520, sigma=20, amp=0.4)
s2 = gaussian(wavelength, mu=600, sigma=40) + gaussian(wavelength, mu=650, sigma=15, amp=0.6)
s3 = gaussian(wavelength, mu=700, sigma=25) + gaussian(wavelength, mu=740, sigma=20, amp=0.5)

# 各スペクトルを正規化（最大値=1）
s1 /= s1.max()
s2 /= s2.max()
s3 /= s3.max()

# ターゲットスペクトル（真の混合比: w1=0.5, w2=0.3, w3=0.2 + ノイズ）
true_weights = np.array([0.5, 0.3, 0.2])
noise = np.random.normal(0, 0.02, size=len(wavelength))
target = true_weights[0]*s1 + true_weights[1]*s2 + true_weights[2]*s3 + noise
target = np.clip(target, 0, None)  # 負値をゼロにクリップ

print('サンプルデータ生成完了')
print(f'  波長点数: {len(wavelength)}')
print(f'  真の混合比: s1={true_weights[0]}, s2={true_weights[1]}, s3={true_weights[2]}')

### 2-B. 実データの読み込み（CSVファイルの場合）
サンプルデータを使う場合はこのセルをスキップしてください。

In [ ]:
# ============================================================
# CSVファイルから読み込む場合の例
# 各CSVの1列目: 波長、2列目: 強度
# ============================================================

# # Google Colab でファイルをアップロードする場合
# from google.colab import files
# uploaded = files.upload()  # ファイル選択ダイアログが開く

# import io
# data1 = np.loadtxt(io.BytesIO(uploaded['spectrum1.csv']), delimiter=',', skiprows=1)
# data2 = np.loadtxt(io.BytesIO(uploaded['spectrum2.csv']), delimiter=',', skiprows=1)
# data3 = np.loadtxt(io.BytesIO(uploaded['spectrum3.csv']), delimiter=',', skiprows=1)
# data_target = np.loadtxt(io.BytesIO(uploaded['target.csv']), delimiter=',', skiprows=1)

# wavelength = data1[:, 0]
# s1 = data1[:, 1]
# s2 = data2[:, 1]
# s3 = data3[:, 1]
# target = data_target[:, 1]

print('（このセルはコメントアウト中。実データを使う場合はコメントを外してください）')

## 3. 入力スペクトルの可視化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 基底スペクトル
ax = axes[0]
ax.plot(wavelength, s1, label='基底スペクトル s1', color='tab:blue')
ax.plot(wavelength, s2, label='基底スペクトル s2', color='tab:orange')
ax.plot(wavelength, s3, label='基底スペクトル s3', color='tab:green')
ax.set_xlabel('波長 (nm)')
ax.set_ylabel('強度')
ax.set_title('基底スペクトル（3つ）')
ax.legend()
ax.grid(True, alpha=0.3)

# ターゲットスペクトル
ax = axes[1]
ax.plot(wavelength, target, label='ターゲット', color='black', lw=2)
ax.set_xlabel('波長 (nm)')
ax.set_ylabel('強度')
ax.set_title('ターゲットスペクトル（近似したいスペクトル）')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. スペクトル近似

行列 `A`（列 = 各基底スペクトル）を構築し、`A @ w ≈ target` を最小化する重み `w` を求めます。

In [ ]:
# 基底スペクトル行列（shape: n_wavelengths × n_basis）
A = np.column_stack([s1, s2, s3])  # ← スペクトルを増減させる場合はここを変更
print(f'行列 A の形状: {A.shape}  （波長点数 × 基底スペクトル数）')

# ----------------------------------------------------------------
# 手法 1: 最小二乗法（制約なし）
# ----------------------------------------------------------------
w_lstsq, residuals, rank, sv = np.linalg.lstsq(A, target, rcond=None)
fitted_lstsq = A @ w_lstsq
rmse_lstsq = np.sqrt(np.mean((fitted_lstsq - target) ** 2))

print('\n--- 手法1: 最小二乗法（制約なし）---')
for i, w in enumerate(w_lstsq):
    print(f'  w{i+1} = {w:.4f}')
print(f'  RMSE = {rmse_lstsq:.6f}')

# ----------------------------------------------------------------
# 手法 2: 非負最小二乗法（NNLS）
# ----------------------------------------------------------------
w_nnls, res_nnls = nnls(A, target)
fitted_nnls = A @ w_nnls
rmse_nnls = np.sqrt(np.mean((fitted_nnls - target) ** 2))

print('\n--- 手法2: 非負最小二乗法（NNLS）---')
for i, w in enumerate(w_nnls):
    print(f'  w{i+1} = {w:.4f}')
print(f'  RMSE = {rmse_nnls:.6f}')

# ----------------------------------------------------------------
# 手法 3: 混合比制約（Σw=1, w≥0）
# ----------------------------------------------------------------
n_basis = A.shape[1]

def objective(w):
    return np.sum((A @ w - target) ** 2)

constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}
bounds = [(0, None)] * n_basis
x0 = np.ones(n_basis) / n_basis  # 初期値：均等配分

result = minimize(objective, x0, method='SLSQP',
                  bounds=bounds, constraints=constraints)
w_mix = result.x
fitted_mix = A @ w_mix
rmse_mix = np.sqrt(np.mean((fitted_mix - target) ** 2))

print('\n--- 手法3: 混合比制約（合計=1、非負）---')
for i, w in enumerate(w_mix):
    print(f'  w{i+1} = {w:.4f}')
print(f'  合計 = {w_mix.sum():.6f}')
print(f'  RMSE = {rmse_mix:.6f}')

## 5. 結果の可視化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

methods = [
    ('最小二乗法\n（制約なし）', fitted_lstsq, w_lstsq, rmse_lstsq),
    ('非負最小二乗法\n（NNLS）', fitted_nnls, w_nnls, rmse_nnls),
    ('混合比制約\n（合計=1、非負）', fitted_mix, w_mix, rmse_mix),
]

for ax, (title, fitted, weights, rmse) in zip(axes, methods):
    ax.plot(wavelength, target, 'k-', lw=2, label='ターゲット', alpha=0.7)
    ax.plot(wavelength, fitted, 'r--', lw=2, label='近似結果')
    ax.fill_between(wavelength, target, fitted,
                    alpha=0.15, color='red', label='残差')

    # 重みを凡例に追加
    weight_str = '  '.join([f'w{i+1}={w:.3f}' for i, w in enumerate(weights)])
    ax.set_title(f'{title}\n{weight_str}\nRMSE={rmse:.4f}', fontsize=10)
    ax.set_xlabel('波長 (nm)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('強度')
plt.suptitle('スペクトル近似結果の比較', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('spectrum_approximation_result.png', bbox_inches='tight', dpi=150)
plt.show()
print('図を spectrum_approximation_result.png として保存しました')

## 6. 重みの比較（棒グラフ）

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(n_basis)
width = 0.22
labels = [f's{i+1}' for i in range(n_basis)]

# 真の重み（サンプルデータの場合のみ表示）
try:
    ax.bar(x - width*1.5, true_weights, width, label='真の重み', color='gray', alpha=0.6)
except NameError:
    pass

ax.bar(x - width*0.5, w_lstsq, width, label='最小二乗法', color='tab:blue')
ax.bar(x + width*0.5, w_nnls,  width, label='NNLS', color='tab:orange')
ax.bar(x + width*1.5, w_mix,   width, label='混合比制約', color='tab:green')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('重み')
ax.set_title('各手法で求めた重みの比較')
ax.legend()
ax.axhline(0, color='black', lw=0.8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. 残差スペクトル

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(wavelength, target - fitted_lstsq, label=f'最小二乗法 (RMSE={rmse_lstsq:.4f})', alpha=0.8)
ax.plot(wavelength, target - fitted_nnls,  label=f'NNLS (RMSE={rmse_nnls:.4f})', alpha=0.8)
ax.plot(wavelength, target - fitted_mix,   label=f'混合比制約 (RMSE={rmse_mix:.4f})', alpha=0.8)
ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.set_xlabel('波長 (nm)')
ax.set_ylabel('残差（ターゲット − 近似）')
ax.set_title('残差スペクトル')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. まとめ・手法の選び方

| 手法 | 特徴 | 適用場面 |
|------|------|---------|
| 最小二乗法 | 制約なし・最速 | 重みが負でも許容できる場合 |
| NNLS | 重みが0以上 | 濃度・強度など物理的に負にならない場合 |
| 混合比制約 | 重みが0以上、合計=1 | 材料の配合比・純粋な混合モデル |

**RMSEが小さいほど近似精度が高い。**  
制約が強いほど近似精度が下がる可能性があるが、物理的な意味を保てます。

## 9. 基底スペクトルの数を変えるには

セル4の `A = np.column_stack([s1, s2, s3])` を変更するだけです。
例えば4つ使う場合：

In [ ]:
# 例: 4つ目の基底スペクトルを追加する場合
s4 = gaussian(wavelength, mu=760, sigma=18) * 0.8
s4 /= s4.max()

A4 = np.column_stack([s1, s2, s3, s4])
w4_nnls, _ = nnls(A4, target)
fitted4 = A4 @ w4_nnls
rmse4 = np.sqrt(np.mean((fitted4 - target) ** 2))

print('4つの基底スペクトルを使った NNLS 結果：')
for i, w in enumerate(w4_nnls):
    print(f'  w{i+1} = {w:.4f}')
print(f'  RMSE = {rmse4:.6f}  （3つの場合: {rmse_nnls:.6f}）')